In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, f1_score
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# CẤU HÌNH CHIẾN THUẬT NÂNG CẤP V2
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Đang chạy CHIẾN THUẬT NÂNG CẤP V2 trên thiết bị: {device}\n")

# ==========================================
# FGM - FAST GRADIENT METHOD (Adversarial Training)
# Giúp mô hình chống overfitting và tăng F1-Score
# ==========================================
class FGM:
    def __init__(self, model):
        self.model = model
        self.backup = {}

    def attack(self, epsilon=1.0, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    r_at = epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self, emb_name='word_embeddings'):
        for name, param in self.model.named_parameters():
            if param.requires_grad and emb_name in name:
                assert name in self.backup
                param.data = self.backup[name]
        self.backup = {}

# ==========================================
# KHỞI TẠO CÁC LỚP HỖ TRỢ (FOCAL LOSS)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean', label_smoothing=0.1):
        super(FocalLoss, self).__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs, targets, weight=self.weight,
            reduction='none', label_smoothing=self.label_smoothing
        )
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean': return focal_loss.mean()
        if self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

# ==========================================
# BƯỚC 2 & 3: ĐỌC DỮ LIỆU VÀ DATALOADER
# ==========================================
print("--- Đọc và xử lý dữ liệu ---")
with open('labeled_results_all_v6.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

def map_label(value):
    if value == 0.0: return 0
    elif value == 0.5: return 1
    elif value == 1.0: return 2
    return 1

processed_data = []
for item in raw_data:
    text = item['original_data']['textTranslated']
    labels = {l['name']: map_label(l['value']) for l in item['labels']}
    processed_data.append({
        'text': text,
        'Food quality': labels.get('Food quality', 1),
        'Price': labels.get('Price', 1),
        'Service quality': labels.get('Service quality', 1),
        'Atmosphere': labels.get('Atmosphere', 1)
    })

df = pd.DataFrame(processed_data)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Atmosphere'])

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class RestaurantReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df['text'].values
        self.aspect_labels = df[['Food quality', 'Price', 'Service quality', 'Atmosphere']].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        labels = self.aspect_labels[index]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

BATCH_SIZE = 32
train_loader = DataLoader(RestaurantReviewDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(RestaurantReviewDataset(val_df, tokenizer), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

# ==========================================
# BƯỚC 4: MÔ HÌNH PHOBERT ABSA (NÂNG CẤP POOLING)
# ==========================================
class PhoBertABSA(nn.Module):
    def __init__(self, n_classes=3, n_aspects=4):
        super(PhoBertABSA, self).__init__()
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base-v2")
        hidden_size = self.phobert.config.hidden_size

        # Concat Mean Pooling và Max Pooling (x2 hidden_size)
        self.fc_pool = nn.Linear(hidden_size * 2, hidden_size)

        self.aspect_projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_size, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.40)
            ) for _ in range(n_aspects)
        ])

        self.dropouts = nn.ModuleList([nn.Dropout(p=0.20) for _ in range(5)])

        self.classifiers = nn.ModuleList([
            nn.Linear(256, n_classes) for _ in range(n_aspects)
        ])

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        # Mean Pooling
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        # Max Pooling
        last_hidden_state[input_mask_expanded == 0] = -1e9  # Đặt giá trị rất nhỏ cho padding
        max_pooled = torch.max(last_hidden_state, 1)[0]

        # Kết hợp Mean và Max Pooling
        pooled_output = torch.cat((mean_pooled, max_pooled), 1)
        pooled_output = F.relu(self.fc_pool(pooled_output))

        logits = []
        for i in range(4):
            aspect_features = self.aspect_projections[i](pooled_output)
            stacked_logits = torch.stack([
                self.classifiers[i](drop(aspect_features)) for drop in self.dropouts
            ], dim=0)
            logits.append(torch.mean(stacked_logits, dim=0))

        return logits

model = PhoBertABSA().to(device)

# ==========================================
# BƯỚC 5: OPTIMIZER & LOSS
# ==========================================
EPOCHS = 30
MAX_LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 7

# Hàm lấy trọng số cân bằng
def get_safe_class_weights(labels_array, aspect_name):
    classes = np.unique(labels_array)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels_array)
    weights = np.clip(weights, a_min=0.5, a_max=5.0) # Giảm trần weight để tránh mô hình bị lệch quá mức

    # Điều chỉnh nhẹ cho Atmosphere và Food
    if aspect_name == 'Atmosphere':
        weights[0] *= 1.2 # Phạt nặng hơn khi đoán sai Tiêu cực
    elif aspect_name == 'Food quality':
        weights[1] *= 1.1

    return torch.tensor(weights, dtype=torch.float).to(device)

aspect_columns = ['Food quality', 'Price', 'Service quality', 'Atmosphere']
loss_fns = []
for col in aspect_columns:
    weights = get_safe_class_weights(train_df[col].values, col)
    # Gamma=2.5 tập trung mạnh hơn vào các mẫu khó đoán (nhầm lẫn Neutral/Positive)
    loss_fns.append(FocalLoss(weight=weights, gamma=2.5, label_smoothing=0.1))

optimizer = AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
# Sử dụng Cosine Annealing để vượt local minima hiệu quả hơn Linear
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# ==========================================
# VÒNG LẶP HUẤN LUYỆN
# ==========================================
fgm = FGM(model)
scaler = torch.amp.GradScaler('cuda')
# ==========================================
# KHỞI TẠO BIẾN LƯU LỊCH SỬ HUẤN LUYỆN
# ==========================================
history = {
    'train_loss': [], 'val_loss': [],
    'train_f1': [], 'val_f1': [], 'val_atmos': []
}

# ==========================================
# BƯỚC 5: VÒNG LẶP HUẤN LUYỆN (Tích hợp Auto-Tracking)
# ==========================================
print("\n🚀 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...")
best_combined_score = 0.0
patience_counter = 0

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS} (LR: {optimizer.param_groups[0]['lr']:.2e})")

    # ---------------- TRAINING ----------------
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0
    all_preds, all_targets = [], []

    for batch in tqdm(train_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad(set_to_none=True)

        # 1. Forward pass chuẩn (Sửa lỗi warning autocast)
        with torch.amp.autocast('cuda'):
            logits = model(input_ids, attention_mask)
            loss = sum([loss_fns[i](logits[i], labels[:, i]) for i in range(4)])
            # Tăng trọng số loss cho Atmosphere
            loss += 1.5 * loss_fns[3](logits[3], labels[:, 3])

        scaler.scale(loss).backward()

        # 2. FGM Attack (Adversarial Training)
        fgm.attack()
        with torch.amp.autocast('cuda'):
            logits_adv = model(input_ids, attention_mask)
            loss_adv = sum([loss_fns[i](logits_adv[i], labels[:, i]) for i in range(4)])
        scaler.scale(loss_adv).backward()
        fgm.restore() # Khôi phục trọng số gốc

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        for i in range(4):
            preds = torch.argmax(logits[i], dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels[:, i].cpu().numpy())

    scheduler.step()
    train_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            with torch.amp.autocast('cuda'):
                logits = model(input_ids, attention_mask)
                v_loss = sum([loss_fns[i](logits[i], labels[:, i]) for i in range(4)])

            val_loss += v_loss.item()

            for i in range(4):
                probs = F.softmax(logits[i], dim=1)
                preds = torch.argmax(logits[i], dim=1)

                # Logic Threshold động giúp hạn chế nhầm lẫn
                if i == 3: # Atmosphere
                    preds[probs[:, 0] >= 0.40] = 0
                    mask_positive = (probs[:, 2] >= 0.40) & (probs[:, 0] < 0.40)
                    preds[mask_positive] = 2
                elif i == 0: # Food
                    mask_food_pos = (probs[:, 2] >= 0.45) & (preds == 1)
                    preds[mask_food_pos] = 2

                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels[:, i].cpu().numpy())

    atmos_preds = [val_preds[idx] for idx in range(len(val_preds)) if (idx % 4) == 3]
    atmos_targets = [val_targets[idx] for idx in range(len(val_targets)) if (idx % 4) == 3]

    val_f1_all = f1_score(val_targets, val_preds, average='macro', zero_division=0)
    val_f1_atmos = f1_score(atmos_targets, atmos_preds, average='macro', zero_division=0)

    avg_train_loss = total_loss/len(train_loader)
    avg_val_loss = val_loss/len(val_loader)

    print(f"Train Loss: {avg_train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f} | Val F1: {val_f1_all:.4f} | Val Atmos F1: {val_f1_atmos:.4f}")

    # LƯU LỊCH SỬ CHO BƯỚC 6
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1_all)
    history['val_atmos'].append(val_f1_atmos)

    # LƯU TRỌNG SỐ TỐT NHẤT
    combined_score = (0.5 * val_f1_all) + (0.5 * val_f1_atmos)
    if combined_score > best_combined_score:
        print(f"🌟 Cải thiện! Điểm tổng hợp: {best_combined_score:.4f} -> {combined_score:.4f}. Lưu best_phobert_absa.pth")
        best_combined_score = combined_score
        torch.save(model.state_dict(), 'best_phobert_absa.pth')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"⏳ Kích hoạt Early Stopping tại Epoch {epoch + 1}!")
            break

# ==========================================
# BƯỚC 6: TỰ ĐỘNG VẼ BIỂU ĐỒ VÀ XUẤT CONFUSION MATRIX
# ==========================================
print("\n" + "="*60)
print("--- BƯỚC 6: VẼ BIỂU ĐỒ & ĐÁNH GIÁ MÔ HÌNH ---")
print("="*60)

# 1. VẼ BIỂU ĐỒ
epochs_range = range(1, len(history['train_loss']) + 1)
plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
plt.plot(epochs_range, history['train_loss'], label='Train Loss', marker='o', linewidth=2)
plt.plot(epochs_range, history['val_loss'], label='Validation Loss', marker='o', linewidth=2)
plt.title('Loss qua các Epoch', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 3, 2)
plt.plot(epochs_range, history['train_f1'], label='Train Macro F1', marker='o', color='purple', linewidth=2)
plt.plot(epochs_range, history['val_f1'], label='Val Macro F1', marker='o', color='red', linewidth=2)
plt.title('Tổng hợp F1-Score (4 Aspects)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 3, 3)
plt.plot(epochs_range, history['val_atmos'], label='Val Atmos F1', marker='o', color='teal', linewidth=2)
plt.title('F1-Score riêng mảng Atmosphere', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 2. XUẤT CONFUSION MATRIX TỪ TRỌNG SỐ TỐT NHẤT
print(f"-> Đang tải trọng số tối ưu từ 'best_phobert_absa.pth' để phân tích...")
model.load_state_dict(torch.load('best_phobert_absa.pth', weights_only=True))
model.eval()

all_preds = {0: [], 1: [], 2: [], 3: []}
all_labels = {0: [], 1: [], 2: [], 3: []}

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Đang đánh giá tập Validation"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast('cuda'):
            logits = model(input_ids, attention_mask)

        for i in range(4):
            preds = torch.argmax(logits[i], dim=1)
            probs = F.softmax(logits[i], dim=1)

            # Phải áp dụng cùng 1 logic như lúc Train
            if i == 3:
                preds[probs[:, 0] >= 0.40] = 0
                mask_positive = (probs[:, 2] >= 0.40) & (probs[:, 0] < 0.40)
                preds[mask_positive] = 2
            elif i == 0:
                mask_food_pos = (probs[:, 2] >= 0.45) & (preds == 1)
                preds[mask_food_pos] = 2

            all_preds[i].extend(preds.cpu().numpy())
            all_labels[i].extend(labels[:, i].cpu().numpy())

class_names = ['Tiêu cực (0)', 'Trung tính (1)', 'Tích cực (2)']
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
axes = axes.flatten()

for i in range(4):
    cm = confusion_matrix(all_labels[i], all_preds[i], labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 12, "weight": "bold"})

    axes[i].set_title(f'Confusion Matrix: {aspect_columns[i]}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Nhãn Dự Đoán', fontsize=11)
    axes[i].set_ylabel('Nhãn Thực Tế', fontsize=11)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print(f"🔥 BẢNG CONFUSION MATRIX ĐỊNH DẠNG MARKDOWN")
print("="*60)

for i in range(4):
    cm = confusion_matrix(all_labels[i], all_preds[i], labels=[0, 1, 2])
    df_cm = pd.DataFrame(
        cm,
        index=[f'Thực tế {c}' for c in class_names],
        columns=[f'Dự đoán {c}' for c in class_names]
    )
    print(f"\n### 🎯 Khía cạnh: {aspect_columns[i]}")
    print(df_cm.to_markdown())


🚀 Đang chạy CHIẾN THUẬT NÂNG CẤP V2 trên thiết bị: cuda

--- Đọc và xử lý dữ liệu ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...

Epoch 1/30 (LR: 2.00e-05)


Validation: 100%|██████████| 140/140 [00:14<00:00,  9.61it/s]


...

Validation: 100%|██████████| 140/140 [00:14<00:00,  9.59it/s]


Train Loss: 0.3360 | Train F1: 0.9998
Val Loss: 0.8563 | Val F1: 0.9097 | Val Atmos F1: 0.9110

Epoch 29/30 (LR: 1.46e-06)


Validation: 100%|██████████| 140/140 [00:14<00:00,  9.61it/s]


Train Loss: 0.3359 | Train F1: 0.9999
Val Loss: 0.8548 | Val F1: 0.9099 | Val Atmos F1: 0.9117

Epoch 30/30 (LR: 1.12e-06)


Training:  11%|█▏        | 63/560 [00:44<05:51,  1.42it/s]